## Window Function in PySpark:

- A Window Function in PySpark is used to `performs calculations over a group of rows without combining them into a single row`. 
- It allows us to `calculate rankings, running totals, averages, and comparisons` while keeping every original row

### Real-time example: 

In a company, HR wants to find the highest-paid employee in each department, rank employees by salary, and calculate each employee's salary compared with the previous employee.

Window functions are commonly used in:

- Employee salary ranking
- Banking transaction analysis
- Sales reports and monthly revenue
- Customer purchase history
- Running totals and moving averages
- Finding the latest record for each customer

### Key difference:

- `groupBy()` combines rows into summary rows.
- `Window functions`performs calculate results across related rows without removing the original rows.


## Create a real-time employee DataFrame:

Imagine an HR department wants to analyze employee salaries, rank employees, and compare salaries within each department.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    row_number, rank, dense_rank,
    lag, lead, sum, avg, min, max,
    first, last, count, ntile
)

spark = SparkSession.builder \
    .appName("WindowFunctions") \
    .getOrCreate()

data = [
    (101, "Arun",   "IT",      60000),
    (102, "Bala",   "IT",      50000),
    (103, "Kumar",  "IT",      50000),
    (104, "Ravi",   "HR",      40000),
    (105, "Priya",  "HR",      45000),
    (106, "Divya",  "HR",      35000),
    (107, "Manoj",  "Finance", 70000),
    (108, "Kavin",  "Finance", 60000),
    (109, "Meena",  "Finance", 60000)
]

columns = [
    "EmployeeID", "Name", "Department", "Salary"
]

employees_df = spark.createDataFrame(data, columns)

employees_df.show() 

### Windows Function used below:

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("WindowFunctions").getOrCreate()

window_spec = Window.partitionBy("Department").orderBy("Salary")

employees_df.withColumn("rank", rank().over(window_spec)).show()

### Understand the Window specification:

A window specification tells Spark `which rows to compare and in what order`.

In [0]:
window_spec=Window.partitionBy("Department").orderBy("Salary")

## PySpark Window Specification Methods:

| Method | Type | Definition | Example |
|---|---|---|---|
| `partitionBy()` | Window method | Divides rows into separate groups for window calculations. | `Window.partitionBy("Department")` |
| `orderBy()` | Window method | Defines the order of rows within each partition. | `.orderBy("Salary")` |
| `rowsBetween()` | Window method | Defines a window frame using physical row positions relative to the current row. | `.rowsBetween(-2, 0)` |
| `rangeBetween()` | Window method | Defines a window frame using the values of the ordering column. | `.rangeBetween(-10000, 0)` |

### Important Notes

- `partitionBy()` creates separate groups for calculations without reducing the number of rows.
- `orderBy()` specifies sorting within the window.
- `rowsBetween(-2, 0)` includes the current row and the previous two rows.
- `rangeBetween(-10000, 0)` includes rows whose ordering values are between the current value minus 10,000 and the current value, inclusive.


## PySpark Window Ranking Functions:

| Function | Definition | Handling Ties |
|---|---|---|
| `row_number()` | Assigns a unique number to each row within a window partition. | Each row gets a different number, even when values are tied. |
| `rank()` | Assigns the rank to rows with equal ordering values. | Skips the next rank after a tie. |
| `dense_rank()` | Assigns the rank to rows with equal ordering values. | Does not skip ranks after a tie. |

### Example

Consider employee salaries sorted in descending order:

| EmployeeName | Salary | `row_number()` | `rank()` | `dense_rank()` |
|---|---:|---:|---:|---:|
| Arun | 50000 | 1 | 1 | 1 |
| Bala | 40000 | 2 | 2 | 2 |
| Kumar | 40000 | 3 | 2 | 2 |
| Ravi | 30000 | 4 | 4 | 3 |

**Interview tip:** To find the second-highest distinct salary, use `dense_rank()` and filter where the rank equals `2`.

Example: `row_number(), rank() and dense_rank()`



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

window_spec=Window.partitionBy("Department").orderBy("Salary")

result_df=employees_df.withColumn("rank",rank().over(window_spec))\
                      .withColumn("dense_rank",dense_rank().over(window_spec))\
                      .withColumn("row_number",row_number().over(window_spec))

result_df.show()

_**\ (backslash)** is used to break the current line and carry on at next line._

### Find the highest-paid employee in each department.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

window_spec=Window.partitionBy("Department")\
            .orderBy(col("Salary").desc(),col("EmployeeID"))

employees_df=employees_df.withColumn("row_number",row_number().over(window_spec))\
            .filter(col("row_number") == 1) \
            .drop("row_number")

employees_df.show()

### lag() and lead() functions:

- `lag()` retrieves a value from a previous row.
- `lead()` retrieves a value from a subsequent row.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

sales_data = [
    (1, "2026-10-20", 10000),
    (1, "2026-10-21", 15000),
    (1, "2026-10-22", 12000),
    (1, "2026-10-23", 18000)
]

sales_df=spark.createDataFrame(sales_data,["store_id","sales_date","revenue"])
sales_df.show()


windows_spec=Window.partitionBy("store_id").orderBy("sales_date")

comparison_df=sales_df.withColumn("previous_revenue",lag("revenue",1).over(windows_spec))\
        .withColumn("next_revenue",lead("revenue",1).over(windows_spec))

comparison_df.show()

**a company can use `lag()` to calculate day-over-day revenue growth.**

In [0]:
from pyspark.sql.functions import when

comparison_df.withColumn(
    "Revenue_Difference",
    col("revenue") - col("previous_revenue")
).withColumn(
    "Growth_Percent",
    when(
        col("previous_revenue").isNotNull()
        & (col("previous_revenue") != 0),
        (
            (col("revenue") - col("previous_revenue"))  / col("previous_revenue")
          
        ) * 100
    )
).show()

### Aggregate window functions:

 Aggregate window functions is used to `calculate the values for each window while retaining individual rows`.

%md
## Common PySpark Window Aggregation Functions

| Function | Definition |
|---|---|
| `sum()` | Calculates the total value across a window. |
| `avg()` | Calculates the average value across a window. |
| `min()` | Returns the minimum value in a window. |
| `max()` | Returns the maximum value in a window. |
| `count()` | Counts rows or non-null values in a window. |

### Important Note:

Here, each employee receives the **total salary, average salary, minimum salary, maximum salary, and employee count** for their department.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

dept_window = Window.partitionBy("Department")

salary_stats_df = employees_df \
    .withColumn("TotalSalary", sum("Salary").over(dept_window)) \
    .withColumn("AverageSalary", avg("Salary").over(dept_window)) \
    .withColumn("MinSalary", min("Salary").over(dept_window)) \
    .withColumn("MaxSalary", max("Salary").over(dept_window)) \
    .withColumn("EmployeeCount", count("*").over(dept_window))

salary_stats_df.select(
    "Name", "Department", "Salary",
    "TotalSalary", "AverageSalary",
    "MinSalary", "MaxSalary", "EmployeeCount"
).orderBy("Department", "Name").show()

Running total using `sum()`

Calculate a running total of employee salaries or cumulative sales over time.

In [0]:
from pyspark.sql.functions import sum, col

running_window = Window \
    .partitionBy("store_id") \
    .orderBy("sales_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_total_df = sales_df.withColumn(
    "RunningTotal",
    sum("revenue").over(running_window)
)

running_total_df.show()

## PySpark Window Frame — rowsBetween()

| Code | Meaning |
|---|---|
| `.rowsBetween()` | Defines which rows are included in a window calculation. |
| `Window.unboundedPreceding` | Starts from the first row in the current window partition. |
| `Window.currentRow` | Ends at the current row, including that row. |

### Example

```python
.rowsBetween(Window.unboundedPreceding, Window.currentRow)
```

**Meaning:** Includes all rows from the beginning of the window partition up to and including the current row.

### Moving average using avg():

A sales team wants the average revenue of the current day and previous two days to smooth out daily fluctuations.

In [0]:
from pyspark.sql.functions import avg

moving_window = Window \
    .partitionBy("store_id") \
    .orderBy("Sales_date") \
    .rowsBetween(-2, 0)

moving_avg_df = sales_df.withColumn(
    "MovingAverage",
    avg("Revenue").over(moving_window)
)

moving_avg_df.show()

### first() and last() window functions:

These functions used to `retrieve the first or last value` in a window frame.

Find the lowest-to-highest salary range for each department and display the first and last salaries in that ordering.

In [0]:
from pyspark.sql.functions import first, last, col

salary_window = Window \
    .partitionBy("Department") \
    .orderBy(col("Salary").asc(), col("EmployeeID")) \
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

salary_range_df = employees_df \
    .withColumn("FirstSalary", first("Salary").over(salary_window)) \
    .withColumn("LastSalary", last("Salary").over(salary_window))

salary_range_df.select(
    "Name", "Department", "Salary",
    "FirstSalary", "LastSalary"
).show()

### ntile():
 — Divide rows into buckets

Divide employees into four salary bands within their department.

In [0]:
from pyspark.sql.functions import ntile, col

salary_window = Window \
    .partitionBy("Department") \
    .orderBy(col("Salary").desc(), col("EmployeeID"))

salary_band_df = employees_df.withColumn(
    "SalaryBand",
    ntile(4).over(salary_window)
)

salary_band_df.orderBy(
    "Department", "SalaryBand", col("Salary").desc()
).show()


%md
## PySpark Window Functions Quick Reference:

| Function | Main Purpose | Real-Time Use |
|---|---|---|
| `row_number()` | Assigns a unique sequential number to each row. | Select one top employee per department. |
| `rank()` | Ranks rows and leaves gaps after ties. | Competition-style rankings. |
| `dense_rank()` | Ranks rows without gaps after ties. | Find the second-highest distinct salary. |
| `lag()` | Retrieves a value from a previous row. | Compare current revenue with previous-day revenue. |
| `lead()` | Retrieves a value from a subsequent row. | Access the next scheduled transaction. |
| `sum()` | Calculates the total over a window. | Calculate cumulative sales. |
| `avg()` | Calculates the average over a window. | Calculate a moving average. |
| `min()` / `max()` | Returns the minimum or maximum value over a window. | Find the lowest and highest salaries. |
| `count()` | Counts rows or non-null values over a window. | Count employees per department. |
| `first()` / `last()` | Returns the first or last value within a window frame. | Retrieve the earliest or latest ordered value. |
| `ntile()` | Divides ordered rows into a specified number of buckets. | Divide employees into salary quartiles. |



### Interview questions:

**Q1. What is a window function in PySpark?**

A window function performs calculations across related rows without collapsing them into a single row per group.

**Q2. What is the difference between groupBy() and a window function?**

groupBy() typically returns one row per group after aggregation, whereas a window function keeps individual rows and adds calculated values.

**Q3. What is the difference between rank() and dense_rank()?**

Both assign the same rank to tied values. rank() leaves gaps after ties, while dense_rank() does not.

**Q4. What is the difference between lag() and lead()?**

lag() retrieves a value from a preceding row; lead() retrieves a value from a following row.

**Q5. Why do we use partitionBy()?**

It separates rows into independent groups so the window calculation runs separately for each group.

**Q6. What is a running total?**

A running total is a cumulative sum that adds values from the beginning of an ordered window up to the current row.

**Q7. What is the purpose of rowsBetween()?**

It defines the physical row boundaries included in a window frame, such as the previous two rows through the current row.

**Q8. Can a window function be used to find the second-highest salary in each department?**

Yes. Apply dense_rank() partitioned by department and ordered by salary descending, then filter for DenseRank == 2. This returns all employees earning the second-highest distinct salary.